# Baseline: Дерево решений

Строим простую интерпретируемую модель — **дерево решений** с `max_depth=4` — на **статичных признаках** (без lag/rolling). Цели:

- Задать нижнюю планку качества (naive baseline)
- Визуализировать логику разбиений датасета
- Убедиться, что переход к сложной модели (CatBoost, ноутбук 03) даёт реальный прирост

Temporal-признаки (lag, rolling, EWM) здесь не используются: их нет при холодном старте, и baseline должен быть честным.

## Импорты и подключение к mlflow

In [ ]:
import os
from datetime import date

from catboost import CatBoostRegressor, Pool
from IPython.display import display
import matplotlib.pyplot as plt
import mlflow
import numpy as np
import optuna
import pandas as pd
import seaborn as sns
import shap
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.tree import DecisionTreeRegressor, export_text, plot_tree

In [47]:
RANDOM_SEED     = 42
N_TRIALS        = 50

mlflow.set_tracking_uri("http://192.168.1.10:5000")
mlflow.set_experiment(f"bikeshare_catboost_{date.today().isoformat()}")

<Experiment: artifact_location='/mlflow-data/artifacts/2', creation_time=1779605603304, experiment_id='2', last_update_time=1779605603304, lifecycle_stage='active', name='bikeshare_catboost_2026-05-24', tags={}, trace_location=None, workspace='default'>

## Загрузка датасета

In [48]:
train = pd.read_csv("datasets/clean/train.csv")
val = pd.read_csv("datasets/clean/validation.csv")
test = pd.read_csv("datasets/clean/test.csv")
test_cold = pd.read_csv("datasets/clean/test_cold.csv")

TARGET = "cnt"
feature_cols = [c for c in train.columns if c != TARGET]

TEMPORAL_COLS = [
    "cnt_lag_1",
    "cnt_lag_3",
    "cnt_lag_6",
    "cnt_lag_12",
    "cnt_lag_24",
    "cnt_rolling_mean_3",
    "cnt_rolling_mean_6",
    "cnt_rolling_mean_12",
    "cnt_rolling_mean_24",
    "cnt_rolling_std_6",
    "cnt_rolling_std_12",
    "cnt_ewm_6h",
]

test_immediate = test.copy()
test_immediate[TEMPORAL_COLS] = np.nan

train_pool = Pool(train[feature_cols], train[TARGET])
val_pool = Pool(val[feature_cols], val[TARGET])
test_pool = Pool(test[feature_cols], test[TARGET])
test_cold_pool = Pool(test_cold[feature_cols], test_cold[TARGET])
test_immediate_pool = Pool(test_immediate[feature_cols], test_immediate[TARGET])

display(
    pd.DataFrame(
        {
            "rows": [
                len(train),
                len(val),
                len(test),
                len(test_cold),
                len(test_immediate),
            ],
            "temporal_nan": [
                df[TEMPORAL_COLS].isna().sum().sum()
                for df in [train, val, test, test_cold, test_immediate]
            ],
        },
        index=["train", "val", "test", "test_cold", "test_immediate"],
    )
)

,rows,temporal_nan
train,8645,15368
val,4358,0
test,4376,0
test_cold,4376,8320
test_immediate,4376,52512


## Метрики

Создам простую функцию сбора метрик

Модель обучается на `log1p(cnt)`, поэтому функция возвращает:
- **RMSLE**: RMSE в log-пространстве, что модель реально минимизирует
- **RMSE / MAE**: в оригинальном пространстве (велосипеды/час), через `expm1`
- **$R^2$**: в log-пространстве (соответствует обучающей цели)

In [49]:
def eval_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    y_pred = np.clip(y_pred, 0.0, None)
    y_true_orig = np.expm1(y_true)
    y_pred_orig = np.expm1(y_pred)
    return {
        "rmsle": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "rmse": float(np.sqrt(mean_squared_error(y_true_orig, y_pred_orig))),
        "mae": float(mean_absolute_error(y_true_orig, y_pred_orig)),
        "r2": float(r2_score(y_true, y_pred)),
    }


def log_metrics(
    y_true: np.ndarray, y_pred: np.ndarray, prefix: str
) -> dict[str, float]:
    metrics = eval_metrics(y_true, y_pred)
    mlflow.log_metrics({f"{prefix}_{k}": v for k, v in metrics.items()})
    return metrics


def predict_orig_scale(
    model: CatBoostRegressor, pool: Pool, y_log: np.ndarray
) -> tuple[np.ndarray, np.ndarray]:
    y_pred_log = np.clip(model.predict(pool), 0.0, None)
    return np.expm1(y_log), np.expm1(y_pred_log)

## Baseline: Дерево решений

Перед тем как переходить к CatBoost, построим простую интерпретируемую модель — **дерево решений** с `max_depth=4`.

**Зачем**:
- Задаёт нижнюю планку качества (naive baseline)
- Показывает, какие признаки достаточно информативны без нелинейных взаимодействий
- Визуализирует логику разбиений — понятна без ML-знаний
- Подтверждает, что переход к сложной модели (CatBoost) даёт реальный прирост

**Ограничения baseline**:
- Обучается только на **статичных признаках** (без lag/rolling): это честное сравнение, поскольку temporal-признаки требуют исторических данных
- Не обрабатывает NaN — пропуски заполняются медианой обучающей выборки
- `max_depth=4` выбран для наглядной визуализации; более глубокое дерево переобучается на train

In [ ]:
NON_TEMPORAL_COLS = [c for c in feature_cols if c not in TEMPORAL_COLS]

train_median = train[NON_TEMPORAL_COLS].median()

X_train_dt = train[NON_TEMPORAL_COLS].fillna(train_median)
X_val_dt   = val[NON_TEMPORAL_COLS].fillna(train_median)
X_test_dt  = test[NON_TEMPORAL_COLS].fillna(train_median)

y_train_dt = train[TARGET].values
y_val_dt   = val[TARGET].values
y_test_dt  = test[TARGET].values

dt = DecisionTreeRegressor(max_depth=4, random_state=RANDOM_SEED)
dt.fit(X_train_dt, y_train_dt)

dt_metrics_val  = eval_metrics(y_val_dt,  dt.predict(X_val_dt))
dt_metrics_test = eval_metrics(y_test_dt, dt.predict(X_test_dt))

print("Дерево решений (max_depth=4, статичные признаки):")
display(pd.DataFrame({
    "val":  dt_metrics_val,
    "test": dt_metrics_test,
}).T.round(4))

### Визуализация дерева решений

Дерево глубины 4 даёт 15 листьев — достаточно для интерпретации, недостаточно для переобучения. Каждый узел показывает: условие разбиения, количество обучающих образцов и среднее предсказание в листе (в log-пространстве).

In [ ]:
fig, ax = plt.subplots(figsize=(28, 10))
plot_tree(
    dt,
    feature_names=NON_TEMPORAL_COLS,
    filled=True,
    rounded=True,
    fontsize=8,
    ax=ax,
    max_depth=4,
    impurity=False,
    precision=2,
)
plt.title("Дерево решений (max_depth=4): структура разбиений", fontsize=14)
plt.tight_layout()
plt.show()

print("\nТекстовое представление дерева:")
print(export_text(dt, feature_names=NON_TEMPORAL_COLS, max_depth=3))

### Важность признаков в дереве решений

Feature importance дерева — доля снижения ошибки (MSE), которую вносит каждый признак. Сравниваем с результатами CatBoost ниже.

In [ ]:
dt_fi = pd.Series(
    dt.feature_importances_, index=NON_TEMPORAL_COLS
).sort_values(ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(20, 6))

dt_fi.plot(kind="barh", ax=axes[0], color="steelblue")
axes[0].set_title("Важность признаков: Дерево решений\n(доля снижения MSE)", fontsize=12)
axes[0].set_xlabel("Важность (0-1)")
axes[0].grid(axis="x", alpha=0.3)

y_true_dt = np.expm1(y_test_dt)
y_pred_dt = np.expm1(dt.predict(X_test_dt))

axes[1].scatter(y_true_dt, y_pred_dt, alpha=0.2, s=6, color="steelblue")
lim = max(y_true_dt.max(), y_pred_dt.max()) * 1.05
axes[1].plot([0, lim], [0, lim], "r--", linewidth=1.5, label="y = x")
axes[1].set_xlabel("Реальное cnt (велосипеды/час)")
axes[1].set_ylabel("Предсказанное cnt")
axes[1].set_title(
    f"Predicted vs Actual: Дерево решений\n"
    f"RMSE={dt_metrics_test['rmse']:.1f}  MAE={dt_metrics_test['mae']:.1f}  R²={dt_metrics_test['r2']:.3f}",
    fontsize=12,
)
axes[1].set_xlim(0, lim)
axes[1].set_ylim(0, lim)
axes[1].grid(alpha=0.3)
axes[1].legend()

plt.suptitle("Baseline — Дерево решений (max_depth=4)", fontsize=14)
plt.tight_layout()
plt.show()

Дерево выявляет те же ключевые признаки, что и EDA: `hr`, `yr`, `temp`, `is_rush_hour`. При этом его структура сразу читается как правила: "если час < 6.5 → низкий спрос; иначе если yr < 0.5 → средний спрос...".

Метрики дерева задают нижнюю планку — **именно на столько** минимально должен превосходить CatBoost (ноутбук `03_catboost.ipynb`) простое правило.